# 個人資訊問答系統 (RAG Application)

### 執行前檢查清單：
1. **啟動 Ollama**：請確保您的 Ollama 服務正在運行 (`ollama serve`)。
2. **下載模型**：
   - 語意嵌入模型：`ollama pull nomic-embed-text` (預設)
   - 對話模型：`ollama pull qwen2.5` (預設)
3. **檢查檔案**：確保 `personal_information` 資料夾內有您的 PDF 文件。

In [1]:
# 執行主程式並開啟對話視窗 (若只想看視覺化，可跳過此格)
%run main.py

c:\Users\max28\anaconda3\envs\selfQuery\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vectorstore from scratch...
Loading documents from: personal_information
  - Loaded 1 files matching *.pdf
  - Loaded 1 files matching *.pdf
  - Loaded 1 files matching *.pptx
  - Loaded 1 files matching *.docx
Loading documents from: 履歷相關資料
  - Loaded 1 files matching *.pdf
Total documents: 75, chunks: 173
Vectorstore created with 173 chunks.
There are 173 vectors with 768 dimensions


D:\LLM\selfQuery\main.py:331: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 繪圖工具與資料準備
此儲存格負責匯入模組、定義工具函式，並**執行 t-SNE 降維運算**。執行此格後，下方的各種視覺化即可快速切換。

In [ ]:
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

def wrap_text(text, width=40):
    """為 Plotly tooltip 加入換行符號"""
    if not text: return ""
    clean_text = text.replace('\n', ' ').strip()
    return "<br>".join([clean_text[i:i+width] for i in range(0, len(clean_text), width)])

if 'vectorstore' not in locals() or vectorstore is None:
    with open("config.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    db_name = config["vector_db"]["db_name"]
    embed_provider = config["providers"]["embed_provider"]
    ollama_model = config["providers"]["ollama_embed_model"]
    embeddings = OpenAIEmbeddings() if embed_provider == "openai" else OllamaEmbeddings(model=ollama_model)
    vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings) if os.path.exists(db_name) else None

if vectorstore:
    print("正在從向量資料庫提取資料並執行降維運算 (這可能需要一點時間)... ")
    collection_data = vectorstore._collection.get(include=['embeddings', 'metadatas', 'documents'])
    embeddings_data = np.array(collection_data['embeddings'])
    metadatas = collection_data['metadatas']
    documents = collection_data['documents']
    sources = [m.get('source', 'Unknown') for m in metadatas]
    unique_sources = list(set(sources))
    
    perp = min(30, len(embeddings_data)-1)
    tsne_2d_model = TSNE(n_components=2, perplexity=perp, init='pca', learning_rate='auto', random_state=42)
    embed_2d = tsne_2d_model.fit_transform(embeddings_data)
    
    tsne_3d_model = TSNE(n_components=3, perplexity=perp, init='pca', learning_rate='auto', random_state=42)
    embed_3d = tsne_3d_model.fit_transform(embeddings_data)
    
    print(f"運算完成！成功處理 {len(embeddings_data)} 個向量片段。")
else:
    print("請先執行主程式建立資料庫。")
    embeddings_data = []

## 視覺化 I：依檔案來源 (File Sources)
觀察不同檔案片段在向量空間中的分佈。

In [ ]:
if len(embeddings_data) > 0:
    # 2D Visualization
    fig = go.Figure()
    for src in unique_sources:
        mask = np.array([s == src for s in sources])
        fig.add_trace(go.Scatter(
            x=embed_2d[mask, 0], y=embed_2d[mask, 1], mode='markers', name=src,
            text=[f"<b>File:</b> {src}<br><b>Content:</b><br>{wrap_text(documents[j][:200])}..." for j in np.where(mask)[0]],
            hoverinfo='text', marker=dict(size=8)
        ))
    fig.update_layout(
        title="2D Visualization by File Source", template="plotly_white", width=1000, height=750,
        legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
        hoverlabel=dict(align="left")
    )
    fig.show()
    
    # 3D Visualization
    fig_3d = go.Figure()
    for src in unique_sources:
        mask = np.array([s == src for s in sources])
        fig_3d.add_trace(go.Scatter3d(
            x=embed_3d[mask, 0], y=embed_3d[mask, 1], z=embed_3d[mask, 2], mode='markers', name=src,
            text=[f"<b>File:</b> {src}<br><b>Content:</b><br>{wrap_text(documents[j][:200])}..." for j in np.where(mask)[0]],
            hoverinfo='text', marker=dict(size=5, opacity=0.8)
        ))
    fig_3d.update_layout(
        title="3D Visualization by File Source", width=1000, height=850,
        scene=dict(aspectmode='cube'),
        legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5),
        hoverlabel=dict(align="left")
    )
    fig_3d.show()
else:
    print("無資料。")

## 視覺化 II：依 AI 自動分群 (K-Means Clusters)
利用 K-Means 自動分類並由 LLM 總結主題。

In [ ]:
NUM_CLUSTERS = 5
if len(embeddings_data) > 0:
    print(f"正在進行 K-Means 分群與 LLM 主題分析...")
    kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(embeddings_data)
    
    summarizer_llm = ChatOllama(model="qwen2.5", temperature=0)
    cluster_themes = {}
    for i in range(NUM_CLUSTERS):
        sample_text = "\n---\n".join([documents[j] for j in range(len(documents)) if cluster_labels[j] == i][:10])
        prompt = ChatPromptTemplate.from_messages([
            ("system", "你是一個專業文本分析助手。用『五個字以內』繁體中文總結核心主題。只需回傳名稱。"),
            ("human", "文件片段：\n{text}")
        ])
        try:
            response = summarizer_llm.invoke(prompt.format(text=sample_text))
            cluster_themes[i] = response.content.strip().replace('「', '').replace('」', '')
        except: cluster_themes[i] = f"Cluster {i}"
    
    # 2D Visualization (Clustered)
    fig_c = go.Figure()
    for i in range(NUM_CLUSTERS):
        mask = (cluster_labels == i)
        theme = cluster_themes[i]
        fig_c.add_trace(go.Scatter(
            x=embed_2d[mask, 0], y=embed_2d[mask, 1], mode='markers', name=f"C{i}: {theme}",
            text=[f"<b>Theme:</b> {theme}<br><b>File:</b> {sources[j]}<br><b>Content:</b><br>{wrap_text(documents[j][:200])}..." for j in np.where(mask)[0]],
            hoverinfo='text', marker=dict(size=8)
        ))
    fig_c.update_layout(
        title="2D Visualization by AI Cluster", template="plotly_white", width=1000, height=750,
        legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
        hoverlabel=dict(align="left")
    )
    fig_c.show()

    # 3D Visualization (Clustered)
    fig_c3 = go.Figure()
    for i in range(NUM_CLUSTERS):
        mask = (cluster_labels == i)
        theme = cluster_themes[i]
        fig_c3.add_trace(go.Scatter3d(
            x=embed_3d[mask, 0], y=embed_3d[mask, 1], z=embed_3d[mask, 2], mode='markers', name=f"C{i}: {theme}",
            text=[f"<b>Theme:</b> {theme}<br><b>File:</b> {sources[j]}<br><b>Content:</b><br>{wrap_text(documents[j][:200])}..." for j in np.where(mask)[0]],
            hoverinfo='text', marker=dict(size=5, opacity=0.8)
        ))
    fig_c3.update_layout(
        title="3D Visualization by AI Cluster", width=1000, height=850,
        scene=dict(aspectmode='cube'),
        legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5),
        hoverlabel=dict(align="left")
    )
    fig_c3.show()
else:
    print("無資料。")